## Dataset & Env setup

1. **Agent dataset**: contains `main.py` and helper folder `orbit_lite/`.
2. **Opponent dataset**: contains opponent `.py` files for self-play and evaluation


In [1]:
import os
import shutil
import sys
from pathlib import Path

BASE_DATA_SLUG = os.environ.get("ORBITLITE_BASE_DATA_SLUG", "orbitlite")
OPPONENTS_DATA_SLUG = os.environ.get("ORBITLITE_OPPONENTS_DATA_SLUG", "orbit-wars-opponents")

BASE_INPUT = Path(f"/kaggle/input/datasets/cunmincut/0906-current-strongest")
OPPONENTS_INPUT = Path(f"/kaggle/input/datasets/cunmincut/opponents")
WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)

# local fallbacks
V4_CANDIDATES = [BASE_INPUT / "main.py", Path("main.py"), Path("/kaggle/working/main.py")]
HELPER_CANDIDATES = [BASE_INPUT / "orbit_lite", Path("orbit_lite"), Path("/kaggle/working/orbit_lite")]
V4_PATH = next((p for p in V4_CANDIDATES if p.exists()), None)
HELPER_SRC = next((p for p in HELPER_CANDIDATES if p.exists() and p.is_dir()), None)

assert V4_PATH is not None, (
    f"OrbitLite main.py not found"
    "with main.py at /kaggle/input/<slug>/main.py, or place main.py in the working directory."
)
assert HELPER_SRC is not None, (
    f"orbit_lite helper folder not found"
    "with orbit_lite/ at /kaggle/input/<slug>/orbit_lite."
)

V4_LOCAL = WORK / "baseline_main.py"
HELPER_LOCAL = WORK / "orbit_lite"
shutil.copy(V4_PATH, V4_LOCAL)
if HELPER_LOCAL.exists():
    shutil.rmtree(HELPER_LOCAL)
shutil.copytree(HELPER_SRC, HELPER_LOCAL)

if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

print(f"base slug: {BASE_DATA_SLUG} -> {BASE_INPUT}")
print(f"opponents slug: {OPPONENTS_DATA_SLUG} -> {OPPONENTS_INPUT}")
print(f"base loaded: {V4_LOCAL} ({V4_LOCAL.stat().st_size:,} bytes)")
print(f"helper copied: {HELPER_LOCAL} ({len(list(HELPER_LOCAL.glob('*.py')))} python files)")


base slug: orbitlite -> /kaggle/input/datasets/cunmincut/0906-current-strongest
opponents slug: orbit-wars-opponents -> /kaggle/input/datasets/cunmincut/opponents
base loaded: /kaggle/working/baseline_main.py (15,014 bytes)
helper copied: /kaggle/working/orbit_lite (13 python files)


In [2]:
import sys, subprocess, glob, os as _os

WHEEL_CANDIDATES = []
WHEEL_CANDIDATES.extend(glob.glob(str(BASE_INPUT / "kaggle_environments-*.whl")))
WHEEL_CANDIDATES.extend(glob.glob("/kaggle/input/orbit-wars-v4-base/kaggle_environments-*.whl"))
WHEEL_CANDIDATES.extend(glob.glob("/kaggle/input/*/kaggle_environments-*.whl"))

if WHEEL_CANDIDATES:
    wheel = sorted(WHEEL_CANDIDATES)[0]
    print(f"Installing {wheel}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", wheel], check=True)
else:
    print("No offline kaggle_environments wheel found; using the preinstalled package.")

import importlib, kaggle_environments
importlib.reload(kaggle_environments)
envs = sorted(_os.listdir(_os.path.join(_os.path.dirname(kaggle_environments.__file__), "envs")))
print(f"kaggle_environments {kaggle_environments.__version__}; orbit_wars available: {'orbit_wars' in envs}")
assert "orbit_wars" in envs, f"orbit_wars not in envs: {envs}"

import torch
print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")

No offline kaggle_environments wheel found; using the preinstalled package.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open

## Baseline opponents for self-play

In [3]:
from pathlib import Path

OPPONENTS_DIR = OPPONENTS_INPUT
assert OPPONENTS_DIR.exists(), (
    f"Opponent dataset not found at {OPPONENTS_DIR}. "
    "Edit OPPONENTS_DATA_SLUG in the setup cell."
)

OPPONENT_ALLOWLIST = {
    "h3b1.py", "hellburner.py", "launch_safety_1039.py", "pascalorbitworkv14.py",
    "suneet.py", "exp30.py", "top_now.py",
    "buddy.py", "multi-focus.py",
}
all_opponent_files = sorted(OPPONENTS_DIR.glob("*.py"))
OPPONENT_PATHS = [str(p) for p in all_opponent_files if p.name in OPPONENT_ALLOWLIST]
if not OPPONENT_PATHS:
    OPPONENT_PATHS = [str(p) for p in all_opponent_files]

print(f"Found {len(OPPONENT_PATHS)} opponents in {OPPONENTS_DIR}:")
for p in OPPONENT_PATHS:
    print(" ", p)

INCLUDE_SELFPLAY = True
if INCLUDE_SELFPLAY:
    OPPONENT_PATHS.append(str(V4_LOCAL))
    print(f"added OrbitLite self-play: {V4_LOCAL}")


Found 9 opponents in /kaggle/input/datasets/cunmincut/opponents:
  /kaggle/input/datasets/cunmincut/opponents/buddy.py
  /kaggle/input/datasets/cunmincut/opponents/exp30.py
  /kaggle/input/datasets/cunmincut/opponents/h3b1.py
  /kaggle/input/datasets/cunmincut/opponents/hellburner.py
  /kaggle/input/datasets/cunmincut/opponents/launch_safety_1039.py
  /kaggle/input/datasets/cunmincut/opponents/multi-focus.py
  /kaggle/input/datasets/cunmincut/opponents/pascalorbitworkv14.py
  /kaggle/input/datasets/cunmincut/opponents/suneet.py
  /kaggle/input/datasets/cunmincut/opponents/top_now.py
added OrbitLite self-play: /kaggle/working/baseline_main.py


## Data collection

We run the agent against 9 opponents × 10 seeds × 2 sides + 1 self-play x 20 seeds x 2 sides = 220 games. Every move is recorded as a shot with three pieces:

- Source = `(src_id, ang, ships)` - the move v4 picked
- Features = the original 24-dims + our selected 4 more
- Label = was this shot ultimately successful / did the agent own the target planet within 10 turns of the projected fleet arrival?

In [4]:
import math, time, multiprocessing as mp
import numpy as np
from kaggle_environments import make

BOARD = 100.0; MAX_SPEED = 6.0

def fleet_speed(ships):
    if ships <= 0: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (math.log(max(ships, 1)) / math.log(1000.0)) ** 1.5

def find_target_via_ray(src_xy, send_angle, planets, ray_horizon=200.0, perp_margin=1.0):
    """Recover the (likely) target planet of a shot from its (src, angle).

    v4 emits actions as (src_id, angle, ships) — the target is implicit. We
    project a ray from src along `angle` and find the closest planet whose
    bounding circle the ray crosses.
    """
    sx, sy = src_xy; fx, fy = math.cos(send_angle), math.sin(send_angle)
    best_pid, best_perp = -1, 1e9
    for p in planets:
        pid, _, px, py, pr, _, _ = p
        pid = int(pid); px = float(px); py = float(py); pr = float(pr)
        dx = px - sx; dy = py - sy
        t = dx * fx + dy * fy
        if t <= 0 or t > ray_horizon: continue
        perp = abs(dx * fy - dy * fx)
        if perp <= pr + perp_margin and perp < best_perp:
            best_perp = perp; best_pid = pid
    return best_pid

def label_outcome(env_steps, target_id, side, arrival_turn, window=10):
    """Label = 1 iff `side` owns `target_id` at any turn in [arrival_turn, arrival_turn+window]."""
    end_t = min(arrival_turn + window, len(env_steps) - 1)
    start_t = min(arrival_turn, end_t)
    for t in range(start_t, end_t + 1):
        s = env_steps[t][side].observation
        if s is None: continue
        for p in s["planets"]:
            if int(p[0]) == target_id and int(p[1]) == side: return 1
    return 0

FEATURE_NAMES = [
    "src_ships", "src_prod", "src_radius",
    "tgt_ships", "tgt_prod", "tgt_radius",
    "tgt_is_self", "tgt_is_neutral", "tgt_is_enemy",
    "ships_sent", "ship_frac", "dist", "eta", "speed",
    "ally_fleet_n", "ally_fleet_ships", "enemy_fleet_n", "enemy_fleet_ships",
    "turn", "my_ships_total", "enemy_ships_total", "ship_delta",
    "my_planets", "enemy_planets",
]
    
FEATURE_DIM = len(FEATURE_NAMES)
def encode_shot(obs, src_id, target_id, ships_sent):
    pdict = {int(p[0]): p for p in obs["planets"]}
    if src_id not in pdict or target_id not in pdict: return None
    src = pdict[src_id]; tgt = pdict[target_id]
    me = int(obs.get("player", 0))
    fleets = obs.get("fleets", [])
    planets = obs["planets"]
    my_ships_total = sum(int(p[5]) for p in planets if int(p[1]) == me)
    enemy_ships_total = sum(int(p[5]) for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    my_planets = sum(1 for p in planets if int(p[1]) == me)
    enemy_planets = sum(1 for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    sx, sy, sr, sships = float(src[2]), float(src[3]), float(src[4]), int(src[5])
    tx, ty, tr, tships = float(tgt[2]), float(tgt[3]), float(tgt[4]), int(tgt[5])
    sprod, tprod = float(src[6]), float(tgt[6])
    dx, dy = tx - sx, ty - sy
    dist = max(math.hypot(dx, dy) - sr - tr, 0.0)
    speed = fleet_speed(ships_sent)
    eta = dist / max(speed, 0.5)
    own_self = 1.0 if int(tgt[1]) == me else 0.0
    own_neutral = 1.0 if int(tgt[1]) < 0 else 0.0
    own_enemy = 1.0 if (int(tgt[1]) >= 0 and int(tgt[1]) != me) else 0.0
    ship_frac = ships_sent / max(sships, 1)
    ally_n = sum(1 for f in fleets if int(f[1]) == me)
    ally_s = sum(int(f[6]) for f in fleets if int(f[1]) == me)
    enemy_n = sum(1 for f in fleets if int(f[1]) != me)
    enemy_s = sum(int(f[6]) for f in fleets if int(f[1]) != me)
    turn = int(obs.get("step", 0))
    return np.array([
        sships/100.0, sprod/5.0, sr/4.0,
        tships/100.0, tprod/5.0, tr/4.0,
        own_self, own_neutral, own_enemy,
        ships_sent/100.0, ship_frac,
        dist/BOARD, eta/60.0, speed/MAX_SPEED,
        ally_n/10.0, ally_s/100.0, enemy_n/10.0, enemy_s/100.0,
        turn/500.0, my_ships_total/200.0, enemy_ships_total/200.0,
        (my_ships_total - enemy_ships_total)/200.0,
        my_planets/20.0, enemy_planets/20.0,
    ], dtype=np.float32)

def collect_one_game(args):
    teacher_path, opponent_path, seed, side, game_id = args
    paths = [teacher_path, opponent_path] if side == 0 else [opponent_path, teacher_path]
    env = make("orbit_wars", configuration={"randomSeed": seed}, debug=False)
    try:
        env.run(paths)
    except Exception as e:
        return [], game_id, str(e)
    rows = []
    for step_idx, st in enumerate(env.steps):
        s = st[side]
        obs = s.observation
        action = s.action or []
        if obs is None or not action: continue
        planets = obs["planets"]
        src_xy = {int(p[0]): (float(p[2]), float(p[3])) for p in planets}
        for mv in action:
            try:
                src_id, ang, ships = int(mv[0]), float(mv[1]), int(mv[2])
            except Exception:
                continue
            if src_id not in src_xy: continue
            tgt_id = find_target_via_ray(src_xy[src_id], ang, planets)
            if tgt_id < 0 or tgt_id == src_id: continue
            tgt_owner = next((int(p[1]) for p in planets if int(p[0]) == tgt_id), -2)
            if tgt_owner == side: continue  # skip own-planet reinforcements (trivially "successful")
            feat = encode_shot(obs, src_id, tgt_id, ships)
            if feat is None: continue
            tx, ty, tr = next(((float(p[2]), float(p[3]), float(p[4])) for p in planets if int(p[0]) == tgt_id), (0,0,0))
            sx, sy = src_xy[src_id]
            sr = next((float(p[4]) for p in planets if int(p[0]) == src_id), 0)
            dist = max(math.hypot(tx-sx, ty-sy) - sr - tr, 0.0)
            speed = fleet_speed(ships)
            eta_turns = max(int(math.ceil(dist / max(speed, 0.5))), 1)
            arrival_turn = step_idx + eta_turns
            label = label_outcome(env.steps, tgt_id, side, arrival_turn, window=10)
            rows.append((feat, label, game_id, step_idx))
    return rows, game_id, None

# build job list
TEACHER = str(V4_LOCAL)
SEEDS = list(range(101, 111))  # 10 seeds (was 6 in v1)
SELFPLAY_SEEDS = list(range(101, 131))  # 30 self-play seeds (was 24 in v1)
jobs = []
gid = 0
for opp in OPPONENT_PATHS:
    for seed in (SELFPLAY_SEEDS if opp == TEACHER else SEEDS):
        for side in (0, 1):
            gid += 1
            jobs.append((TEACHER, opp, seed, side, gid))
print(f"Jobs: {len(jobs)} games (teacher=orbitlite, {len(OPPONENT_PATHS)} opps, {len(SEEDS)} seeds × 2 sides)")

Jobs: 240 games (teacher=orbitlite, 10 opps, 10 seeds × 2 sides)


In [ ]:
import math
import time
import numpy as np
from pathlib import Path
from kaggle_environments import make


PROPOSED_FEATURE_NAMES = [
    # Local pressure / map geometry.
    "nearest_my_to_target",
    "nearest_enemy_to_target",
    "ally_ships_near_target",
    "enemy_ships_near_target",
    "ally_ships_near_source",
    "enemy_ships_near_source",
    # Capture margin / source safety.
    "capture_margin_growth",
    "target_growth_until_eta",
    "post_launch_source_reserve",
    "send_to_target_ships",
    # Global economy shares.
    "my_ship_share",
    "my_prod_share",
    "my_planet_share",
    "target_prod_share",
    # Angle quality.
    "cos_shot_error",
    "abs_sin_shot_error",
]


def _nearest_planet_distance(planets, x, y, owner_pred):
    vals = []
    for p in planets:
        if owner_pred(int(p[1])):
            vals.append(max(math.hypot(float(p[2]) - x, float(p[3]) - y) - float(p[4]), 0.0))
    return min(vals) if vals else BOARD


def _ships_near_planets(planets, x, y, me, radius=25.0):
    ally = 0.0
    enemy = 0.0
    for p in planets:
        d = math.hypot(float(p[2]) - x, float(p[3]) - y)
        if d > radius:
            continue
        owner = int(p[1])
        ships = float(p[5])
        if owner == me:
            ally += ships
        elif owner >= 0:
            enemy += ships
    return ally, enemy


def encode_proposed_features(obs, src_id, target_id, ships_sent, shot_angle):
    """Return proposed feature vector for one shot, or None if target recovery failed."""
    pdict = {int(p[0]): p for p in obs["planets"]}
    if src_id not in pdict or target_id not in pdict:
        return None

    planets = obs["planets"]
    src = pdict[src_id]
    tgt = pdict[target_id]
    me = int(obs.get("player", 0))

    sx, sy, sr, sships = float(src[2]), float(src[3]), float(src[4]), float(src[5])
    tx, ty, tr, tships = float(tgt[2]), float(tgt[3]), float(tgt[4]), float(tgt[5])
    sprod, tprod = float(src[6]), float(tgt[6])
    dx, dy = tx - sx, ty - sy
    dist = max(math.hypot(dx, dy) - sr - tr, 0.0)
    speed = fleet_speed(ships_sent)
    eta = dist / max(speed, 0.5)

    my_ships = sum(float(p[5]) for p in planets if int(p[1]) == me)
    enemy_ships = sum(float(p[5]) for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    my_prod = sum(float(p[6]) for p in planets if int(p[1]) == me)
    enemy_prod = sum(float(p[6]) for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    my_planets = sum(1 for p in planets if int(p[1]) == me)
    enemy_planets = sum(1 for p in planets if int(p[1]) >= 0 and int(p[1]) != me)

    nearest_my_to_target = _nearest_planet_distance(planets, tx, ty, lambda owner: owner == me)
    nearest_enemy_to_target = _nearest_planet_distance(
        planets, tx, ty, lambda owner: owner >= 0 and owner != me
    )
    ally_near_t, enemy_near_t = _ships_near_planets(planets, tx, ty, me)
    ally_near_s, enemy_near_s = _ships_near_planets(planets, sx, sy, me)

    target_owner = int(tgt[1])
    target_growth_until_eta = 0.0 if target_owner == me else tprod * max(eta, 0.0)
    capture_margin_growth = ships_sent - (tships + target_growth_until_eta)
    post_launch_source_reserve = max(sships - ships_sent, 0.0)
    send_to_target_ships = ships_sent / max(tships + 1.0, 1.0)

    total_ships = my_ships + enemy_ships
    total_prod = my_prod + enemy_prod
    total_planets = my_planets + enemy_planets

    target_angle = math.atan2(dy, dx)
    err = shot_angle - target_angle
    cos_shot_error = math.cos(err)
    abs_sin_shot_error = abs(math.sin(err))

    return np.array(
        [
            nearest_my_to_target / BOARD,
            nearest_enemy_to_target / BOARD,
            ally_near_t / 200.0,
            enemy_near_t / 200.0,
            ally_near_s / 200.0,
            enemy_near_s / 200.0,
            capture_margin_growth / 100.0,
            target_growth_until_eta / 100.0,
            post_launch_source_reserve / 100.0,
            send_to_target_ships,
            my_ships / max(total_ships, 1.0),
            my_prod / max(total_prod, 1.0),
            my_planets / max(total_planets, 1.0),
            tprod / max(total_prod, 1.0),
            cos_shot_error,
            abs_sin_shot_error,
        ],
        dtype=np.float32,
    )


def _auc_1d(x, y):
    """Small dependency-free ROC AUC for one feature."""
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    pos = y == 1
    neg = y == 0
    n_pos = int(pos.sum())
    n_neg = int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan

    order = np.argsort(x)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(x) + 1)

    # Average tied ranks.
    xs = x[order]
    start = 0
    while start < len(x):
        end = start + 1
        while end < len(x) and xs[end] == xs[start]:
            end += 1
        if end - start > 1:
            avg = 0.5 * (start + 1 + end)
            ranks[order[start:end]] = avg
        start = end

    auc = (ranks[pos].sum() - n_pos * (n_pos + 1) / 2.0) / max(1, n_pos * n_neg)
    return float(max(auc, 1.0 - auc))  # direction-free usefulness


def collect_preflight_one_game(args):
    teacher_path, opponent_path, seed, side, game_id = args
    paths = [teacher_path, opponent_path] if side == 0 else [opponent_path, teacher_path]
    env = make("orbit_wars", configuration={"randomSeed": seed}, debug=False)
    try:
        env.run(paths)
    except Exception as e:
        return [], game_id, str(e)

    rows = []
    for step_idx, st in enumerate(env.steps):
        obs = st[side].observation
        action = st[side].action or []
        if obs is None or not action:
            continue
        planets = obs["planets"]
        src_xy = {int(p[0]): (float(p[2]), float(p[3])) for p in planets}
        for mv in action:
            try:
                src_id, ang, ships = int(mv[0]), float(mv[1]), int(mv[2])
            except Exception:
                continue
            if src_id not in src_xy:
                continue
            tgt_id = find_target_via_ray(src_xy[src_id], ang, planets)
            if tgt_id < 0 or tgt_id == src_id:
                continue
            tgt_owner = next((int(p[1]) for p in planets if int(p[0]) == tgt_id), -2)
            if tgt_owner == side:
                continue

            proposed = encode_proposed_features(obs, src_id, tgt_id, ships, ang)
            if proposed is None:
                continue

            tx, ty, tr = next(
                ((float(p[2]), float(p[3]), float(p[4])) for p in planets if int(p[0]) == tgt_id),
                (0, 0, 0),
            )
            sx, sy = src_xy[src_id]
            sr = next((float(p[4]) for p in planets if int(p[0]) == src_id), 0)
            dist = max(math.hypot(tx - sx, ty - sy) - sr - tr, 0.0)
            eta_turns = max(int(math.ceil(dist / max(fleet_speed(ships), 0.5))), 1)
            label = label_outcome(env.steps, tgt_id, side, step_idx + eta_turns, window=10)
            rows.append((proposed, label, game_id, step_idx))
    return rows, game_id, None


# Tiny pilot: change these knobs if you want a stronger preflight signal.
PREFLIGHT_SEEDS = list(range(701, 703))
PREFLIGHT_OPPONENTS = OPPONENT_PATHS[: min(3, len(OPPONENT_PATHS))]
PREFLIGHT_INCLUDE_SELFPLAY = True

preflight_jobs = []
gid = 0
for opp in PREFLIGHT_OPPONENTS:
    for seed in PREFLIGHT_SEEDS:
        for side in (0, 1):
            gid += 1
            preflight_jobs.append((TEACHER, opp, seed, side, gid))
if PREFLIGHT_INCLUDE_SELFPLAY:
    for seed in PREFLIGHT_SEEDS:
        for side in (0, 1):
            gid += 1
            preflight_jobs.append((TEACHER, TEACHER, seed + 1000, side, gid))

print(f"Preflight jobs: {len(preflight_jobs)} games")
t0 = time.time()
preflight_rows = []
failed = 0
for args in preflight_jobs:
    rows, gid_, err = collect_preflight_one_game(args)
    if err is not None:
        failed += 1
        print(f"  [WARN] game {gid_} failed: {err[:80]}")
    else:
        preflight_rows.extend(rows)

print(f"Collected {len(preflight_rows)} preflight shots in {time.time() - t0:.1f}s ({failed} failed games)")
assert preflight_rows, "No preflight rows collected; check opponents/env setup."

Xpf = np.stack([r[0] for r in preflight_rows]).astype(np.float32)
ypf = np.asarray([r[1] for r in preflight_rows], dtype=np.int64)
print(f"positive rate: {ypf.mean() * 100:.1f}%")

summary = []
for j, name in enumerate(PROPOSED_FEATURE_NAMES):
    x = Xpf[:, j]
    pos_x = x[ypf == 1]
    neg_x = x[ypf == 0]
    std = float(np.std(x))
    mean_gap = float(np.mean(pos_x) - np.mean(neg_x)) if len(pos_x) and len(neg_x) else np.nan
    corr = float(np.corrcoef(x, ypf)[0, 1]) if std > 1e-8 and np.std(ypf) > 1e-8 else np.nan
    auc = _auc_1d(x, ypf)
    summary.append(
        {
            "feature": name,
            "std": std,
            "abs_corr": abs(corr) if np.isfinite(corr) else np.nan,
            "auc_dirfree": auc,
            "mean_gap": mean_gap,
        }
    )

corr_mat = np.corrcoef(Xpf, rowvar=False)
for j, row in enumerate(summary):
    others = np.delete(np.abs(corr_mat[j]), j)
    row["max_abs_corr_with_proposed"] = float(np.nanmax(others)) if len(others) else 0.0

summary = sorted(
    summary,
    key=lambda r: (
        0 if np.isnan(r["auc_dirfree"]) else r["auc_dirfree"],
        0 if np.isnan(r["abs_corr"]) else r["abs_corr"],
        r["std"],
    ),
    reverse=True,
)

print("\nProposed feature preflight ranking:")
print("feature                      std      |corr|   auc*    gap      max_redund")
for r in summary:
    keep_hint = (
        "KEEP"
        if r["std"] > 1e-4 and (r["auc_dirfree"] >= 0.56 or r["abs_corr"] >= 0.06) and r["max_abs_corr_with_proposed"] < 0.97
        else "WATCH"
    )
    print(
        f"{r['feature']:<28} {r['std']:>7.4f} "
        f"{r['abs_corr']:>7.3f} {r['auc_dirfree']:>7.3f} "
        f"{r['mean_gap']:>8.4f} {r['max_abs_corr_with_proposed']:>10.3f}  {keep_hint}"
    )

print("\n* auc is direction-free: 0.50 means random; larger means the feature separates success/failure in either direction.")
print("Rule of thumb: KEEP if std is nontrivial, auc >= 0.56 or |corr| >= 0.06, and redundancy < 0.97.")



Preflight jobs: 16 games
Collected 1378 preflight shots in 148.6s (0 failed games)
positive rate: 52.8%

Proposed feature preflight ranking:
feature                      std      |corr|   auc*    gap      max_redund
target_prod_share             0.1165   0.157   0.707   0.0367      0.448  KEEP
my_planet_share               0.1320   0.296   0.652   0.0783      0.986  WATCH
my_prod_share                 0.1360   0.301   0.651   0.0820      0.986  WATCH
nearest_my_to_target          0.1648   0.264   0.635  -0.0870      0.454  KEEP
nearest_enemy_to_target       0.2035   0.017   0.624  -0.0068      0.448  KEEP
my_ship_share                 0.1854   0.240   0.617   0.0890      0.846  KEEP
send_to_target_ships          5.2952   0.130   0.612   1.3822      0.176  KEEP
capture_margin_growth         1.9253   0.074   0.598   0.2860      0.645  KEEP
ally_ships_near_source        0.1882   0.140   0.587  -0.0526      0.161  KEEP
ally_ships_near_target        0.1201   0.031   0.569  -0.0075      0.45

In [5]:
COMPACT_EXTRA_FEATURE_NAMES = [
    "target_prod_share",
    "my_ship_share",
    "send_to_target_ships",
    "capture_margin_growth",
]

REMOVED_BASE_FEATURE_NAMES = {
    "tgt_is_self",
    "speed",
    "my_ships_total",
    "my_planets",
    "enemy_planets",
}

BASE_FEATURE_NAMES_24 = [
    "src_ships", "src_prod", "src_radius",
    "tgt_ships", "tgt_prod", "tgt_radius",
    "tgt_is_self", "tgt_is_neutral", "tgt_is_enemy",
    "ships_sent", "ship_frac", "dist", "eta", "speed",
    "ally_fleet_n", "ally_fleet_ships", "enemy_fleet_n", "enemy_fleet_ships",
    "turn", "my_ships_total", "enemy_ships_total", "ship_delta",
    "my_planets", "enemy_planets",
]
KEPT_BASE_FEATURE_NAMES = [
    name for name in BASE_FEATURE_NAMES_24
    if name not in REMOVED_BASE_FEATURE_NAMES
]
FEATURE_NAMES = KEPT_BASE_FEATURE_NAMES + COMPACT_EXTRA_FEATURE_NAMES
FEATURE_DIM = len(FEATURE_NAMES)

def encode_shot(obs, src_id, target_id, ships_sent):
    """23 dims: 19 retained baseline features + 4 cheap extra features."""
    pdict = {int(p[0]): p for p in obs["planets"]}
    if src_id not in pdict or target_id not in pdict:
        return None

    src = pdict[src_id]
    tgt = pdict[target_id]
    me = int(obs.get("player", 0))
    fleets = obs.get("fleets", [])
    planets = obs["planets"]

    # One planet pass computes all global totals needed by retained/extra features.
    my_ships_total = 0.0
    enemy_ships_total = 0.0
    my_prod_total = 0.0
    enemy_prod_total = 0.0
    for p in planets:
        owner = int(p[1])
        ships = float(p[5])
        prod = float(p[6])
        if owner == me:
            my_ships_total += ships
            my_prod_total += prod
        elif owner >= 0:
            enemy_ships_total += ships
            enemy_prod_total += prod

    sx, sy, sr, sships = float(src[2]), float(src[3]), float(src[4]), int(src[5])
    tx, ty, tr, tships = float(tgt[2]), float(tgt[3]), float(tgt[4]), int(tgt[5])
    sprod, tprod = float(src[6]), float(tgt[6])
    dx, dy = tx - sx, ty - sy
    dist = max(math.hypot(dx, dy) - sr - tr, 0.0)
    speed = fleet_speed(ships_sent)
    eta = dist / max(speed, 0.5)

    own_neutral = 1.0 if int(tgt[1]) < 0 else 0.0
    own_enemy = 1.0 if (int(tgt[1]) >= 0 and int(tgt[1]) != me) else 0.0
    ship_frac = ships_sent / max(sships, 1)

    # One fleet pass replaces four separate scans.
    ally_n = 0
    ally_s = 0.0
    enemy_n = 0
    enemy_s = 0.0
    for f in fleets:
        if int(f[1]) == me:
            ally_n += 1
            ally_s += float(f[6])
        else:
            enemy_n += 1
            enemy_s += float(f[6])
    turn = int(obs.get("step", 0))

    total_ships = my_ships_total + enemy_ships_total
    total_prod = my_prod_total + enemy_prod_total
    target_growth_until_eta = 0.0 if int(tgt[1]) == me else tprod * max(eta, 0.0)

    extra = [
        tprod / max(total_prod, 1.0),
        my_ships_total / max(total_ships, 1.0),
        ships_sent / max(tships + 1.0, 1.0),
        (ships_sent - (tships + target_growth_until_eta)) / 100.0,
    ]

    values = np.array(
        [
            sships / 100.0,
            sprod / 5.0,
            sr / 4.0,
            tships / 100.0,
            tprod / 5.0,
            tr / 4.0,
            own_neutral,
            own_enemy,
            ships_sent / 100.0,
            ship_frac,
            dist / BOARD,
            eta / 60.0,
            ally_n / 10.0,
            ally_s / 100.0,
            enemy_n / 10.0,
            enemy_s / 100.0,
            turn / 500.0,
            enemy_ships_total / 200.0,
            (my_ships_total - enemy_ships_total) / 200.0,
            *extra,
        ],
        dtype=np.float32,
    )
    assert values.shape[0] == FEATURE_DIM, (values.shape, FEATURE_DIM)
    return values


print(f"Using compact feature set: {len(BASE_FEATURE_NAMES_24)} -> {FEATURE_DIM} dims")
print("Removed:", ", ".join(sorted(REMOVED_BASE_FEATURE_NAMES)))
print("Added:", ", ".join(COMPACT_EXTRA_FEATURE_NAMES))

Using compact feature set: 24 -> 23 dims
Removed: enemy_planets, my_planets, my_ships_total, speed, tgt_is_self
Added: target_prod_share, my_ship_share, send_to_target_ships, capture_margin_growth


In [6]:
# run jobs in parallel
all_rows = []
failed = 0
t0 = time.time()
with mp.Pool(processes=4) as pool:
    for i, (rows, gid_, err) in enumerate(pool.imap_unordered(collect_one_game, jobs)):
        if err is not None:
            failed += 1
            print(f"  [WARN] game {gid_} failed: {err[:80]}")
        else:
            all_rows.extend(rows)
        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(jobs)} games, rows={len(all_rows)}, t={time.time()-t0:.0f}s", flush=True)

print(f"\nDone: {len(all_rows)} shots collected ({failed} games failed)")
feats = np.stack([r[0] for r in all_rows]).astype(np.float32)
labels = np.asarray([r[1] for r in all_rows], dtype=np.float32)
meta_game = np.asarray([r[2] for r in all_rows], dtype=np.int32)
pos_rate = labels.mean()
print(f"  features: {feats.shape}, labels: {labels.shape}")
print(f"  positive rate: {pos_rate*100:.1f}%")

if pos_rate > 0.85:
    print(f"  WARNING: pos_rate is high ({pos_rate*100:.1f}%). Validator may be over-cautious.")
    print(f"           Consider stronger opponents or more self-play games.")
elif pos_rate < 0.40:
    print(f"  WARNING: pos_rate is low ({pos_rate*100:.1f}%). Opponents may be too strong.")

# Save dataset for inspection
np.savez_compressed(WORK / "shot_dataset.npz", features=feats, labels=labels.astype(np.int64), meta_game=meta_game, feature_names=np.asarray(FEATURE_NAMES))
print(f"  saved {WORK}/shot_dataset.npz")

  5/240 games, rows=427, t=66s
  10/240 games, rows=837, t=124s
  15/240 games, rows=1425, t=167s
  20/240 games, rows=1795, t=206s
  25/240 games, rows=2120, t=232s
  30/240 games, rows=2439, t=265s
  35/240 games, rows=2806, t=286s
  40/240 games, rows=3121, t=301s
  45/240 games, rows=3403, t=322s
  50/240 games, rows=3665, t=349s
  55/240 games, rows=3892, t=365s
  60/240 games, rows=4219, t=389s
  65/240 games, rows=4419, t=409s
  70/240 games, rows=4593, t=422s
  75/240 games, rows=4832, t=442s
  80/240 games, rows=5028, t=460s
  85/240 games, rows=5199, t=482s
  90/240 games, rows=5393, t=500s
  95/240 games, rows=5560, t=529s
  100/240 games, rows=5837, t=579s
  105/240 games, rows=6281, t=635s
  110/240 games, rows=6674, t=713s
  115/240 games, rows=7205, t=784s
  120/240 games, rows=7816, t=854s
  125/240 games, rows=8148, t=882s
  130/240 games, rows=8298, t=915s
  135/240 games, rows=8481, t=945s
  140/240 games, rows=8605, t=978s
  145/240 games, rows=8800, t=1010s
  150/2

In [18]:
!rm -rf /kaggle/working/weights.npz

## Train the validator


In [15]:
import torch
import torch.nn as nn

class ShotValidator(nn.Module):
    def __init__(self, in_dim=FEATURE_DIM, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def _select_device():
    if not torch.cuda.is_available():
        return torch.device("cpu"), "cuda not available"
    try:
        _ = torch.zeros(4, device="cuda") + 1
        torch.cuda.synchronize()
        return torch.device("cuda"), "cuda OK"
    except Exception as e:
        return torch.device("cpu"), f"cuda failed ({type(e).__name__}: {str(e)[:60]}), using CPU"

device, why = _select_device()
print(f"Training on: {device} ({why})")

rng = np.random.default_rng(42)
games = np.unique(meta_game)
rng.shuffle(games)
n_val = max(1, int(len(games) * 0.2))
val_games = set(games[:n_val].tolist())
val_mask = np.array([g in val_games for g in meta_game], dtype=bool)
Xt, yt = feats[~val_mask], labels[~val_mask]
Xv, yv = feats[val_mask], labels[val_mask]
print(f"  train: {len(Xt)} shots ({len(games)-n_val} games), val: {len(Xv)} shots ({n_val} games)")
print(f"  train pos: {yt.mean()*100:.1f}%, val pos: {yv.mean()*100:.1f}%")

pr = max(yt.mean(), 1e-6)
pos_weight = torch.tensor([(1.0 - pr) / pr], device=device)
print(f"  pos_weight (neg/pos): {pos_weight.item():.3f}")

Training on: cpu (cuda not available)
  train: 14886 shots (192 games), val: 3315 shots (48 games)
  train pos: 52.5%, val pos: 56.6%
  pos_weight (neg/pos): 0.905


In [19]:
# training loop
model = ShotValidator(in_dim=FEATURE_DIM, hidden=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

Xt_t = torch.from_numpy(Xt).to(device)
yt_t = torch.from_numpy(yt).to(device)
Xv_t = torch.from_numpy(Xv).to(device)
yv_t = torch.from_numpy(yv).to(device)

EPOCHS = 40
BATCH = 512

best_acc03 = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    idx = torch.randperm(len(Xt_t), device=device)
    losses = []
    for i in range(0, len(idx), BATCH):
        b = idx[i:i+BATCH]
        logits = model(Xt_t[b])
        loss = crit(logits, yt_t[b])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss))
    tl = float(np.mean(losses))
    model.eval()
    with torch.no_grad():
        vlogits = model(Xv_t)
        vloss = float(crit(vlogits, yv_t))
        vprob = torch.sigmoid(vlogits)
        v_acc05 = float(((vprob > 0.5).float() == yv_t).float().mean())
        v_acc03 = float(((vprob > 0.3).float() == yv_t).float().mean())
    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f"epoch {epoch:3d} | t_loss={tl:.4f} v_loss={vloss:.4f} "
              f"v_acc(0.5)={v_acc05:.3f} v_acc(0.3)={v_acc03:.3f}")

print("Training done.")

epoch   1 | t_loss=0.6732 v_loss=0.6406 v_acc(0.5)=0.616 v_acc(0.3)=0.566
epoch   5 | t_loss=0.5080 v_loss=0.4814 v_acc(0.5)=0.774 v_acc(0.3)=0.707
epoch  10 | t_loss=0.4531 v_loss=0.4365 v_acc(0.5)=0.783 v_acc(0.3)=0.767
epoch  15 | t_loss=0.4253 v_loss=0.4704 v_acc(0.5)=0.766 v_acc(0.3)=0.769
epoch  20 | t_loss=0.4076 v_loss=0.3992 v_acc(0.5)=0.800 v_acc(0.3)=0.790
epoch  25 | t_loss=0.3987 v_loss=0.3906 v_acc(0.5)=0.802 v_acc(0.3)=0.801
epoch  30 | t_loss=0.3950 v_loss=0.3864 v_acc(0.5)=0.808 v_acc(0.3)=0.791
epoch  35 | t_loss=0.3825 v_loss=0.3740 v_acc(0.5)=0.816 v_acc(0.3)=0.802
epoch  40 | t_loss=0.3820 v_loss=0.3722 v_acc(0.5)=0.817 v_acc(0.3)=0.805
Training done.


## Feature Importance

We use permutation importance on the game-level validation split. Higher delta_loss means the validator relies more on that feature.


In [10]:
# Permutation feature importance on validation split.
model.eval()
with torch.no_grad():
    base_logits = model(Xv_t)
    base_loss = float(crit(base_logits, yv_t))
    base_prob = torch.sigmoid(base_logits)
    base_acc03 = float(((base_prob > 0.3).float() == yv_t).float().mean())

rng = np.random.default_rng(123)
importance_rows = []
for j, name in enumerate(FEATURE_NAMES):
    Xp = Xv.copy()
    Xp[:, j] = rng.permutation(Xp[:, j])
    Xp_t = torch.from_numpy(Xp).to(device)
    with torch.no_grad():
        logits = model(Xp_t)
        loss = float(crit(logits, yv_t))
        prob = torch.sigmoid(logits)
        acc03 = float(((prob > 0.3).float() == yv_t).float().mean())
    importance_rows.append({
        "feature": name,
        "delta_loss": loss - base_loss,
        "delta_acc03": base_acc03 - acc03,
    })

importance_rows = sorted(importance_rows, key=lambda r: (r["delta_loss"], r["delta_acc03"]), reverse=True)
print(f"Baseline val loss={base_loss:.4f}, acc@0.3={base_acc03:.3f}")
print("Top permutation importances:")
for r in importance_rows[:20]:
    print(f"  {r['feature']:<24} delta_loss={r['delta_loss']:+.5f} delta_acc03={r['delta_acc03']:+.4f}")

try:
    import pandas as pd
    importance_df = pd.DataFrame(importance_rows)
    importance_df.to_csv(WORK / "feature_importance.csv", index=False)
    display(importance_df.head(20))
    print(f"saved {WORK}/feature_importance.csv")
except Exception as e:
    print(f"pandas/display unavailable ({type(e).__name__}); printed table above only")


Baseline val loss=0.3556, acc@0.3=0.809
Top permutation importances:
  enemy_fleet_ships        delta_loss=+0.16382 delta_acc03=+0.0518
  tgt_is_enemy             delta_loss=+0.15152 delta_acc03=+0.1146
  ally_fleet_ships         delta_loss=+0.12602 delta_acc03=+0.0289
  tgt_prod                 delta_loss=+0.07264 delta_acc03=+0.0593
  tgt_is_neutral           delta_loss=+0.06658 delta_acc03=+0.0736
  dist                     delta_loss=+0.06090 delta_acc03=+0.0344
  ship_delta               delta_loss=+0.05111 delta_acc03=+0.0103
  send_to_target_ships     delta_loss=+0.03130 delta_acc03=+0.0188
  enemy_fleet_n            delta_loss=+0.02973 delta_acc03=+0.0168
  tgt_radius               delta_loss=+0.02959 delta_acc03=+0.0297
  capture_margin_growth    delta_loss=+0.02877 delta_acc03=+0.0085
  tgt_ships                delta_loss=+0.02667 delta_acc03=+0.0241
  ships_sent               delta_loss=+0.01273 delta_acc03=+0.0033
  ship_frac                delta_loss=+0.01170 delta_acc03=+

,feature,delta_loss,delta_acc03
0,enemy_fleet_ships,0.163821,0.051772
1,tgt_is_enemy,0.151518,0.114602
2,ally_fleet_ships,0.126015,0.028902
3,tgt_prod,0.072638,0.059311
4,tgt_is_neutral,0.066575,0.073637
5,dist,0.060896,0.034431
6,ship_delta,0.051113,0.010304
7,send_to_target_ships,0.031296,0.018849
8,enemy_fleet_n,0.029726,0.016838
9,tgt_radius,0.029589,0.029656


saved /kaggle/working/feature_importance.csv


## Packaging


In [20]:
# export weights to numpy
model.eval()
sd = model.state_dict()
weights = {
    "l0_w": sd["net.0.weight"].cpu().numpy().astype(np.float32),
    "l0_b": sd["net.0.bias"].cpu().numpy().astype(np.float32),
    "l1_w": sd["net.2.weight"].cpu().numpy().astype(np.float32),
    "l1_b": sd["net.2.bias"].cpu().numpy().astype(np.float32),
    "l2_w": sd["net.4.weight"].cpu().numpy().astype(np.float32),
    "l2_b": sd["net.4.bias"].cpu().numpy().astype(np.float32),
}
for k, v in weights.items():
    print(f"  {k}: {v.shape}, {v.dtype}")

weights_path = WORK / "weights.npz"
np.savez(weights_path, **weights)
print(f"\nSaved {weights_path} ({weights_path.stat().st_size:,} bytes)")

# sanity check: parity between pytorch and numpy forward
def numpy_forward(x, W):
    h = np.maximum(0, x @ W["l0_w"].T + W["l0_b"])
    h = np.maximum(0, h @ W["l1_w"].T + W["l1_b"])
    logit = h @ W["l2_w"].T + W["l2_b"]
    return 1.0 / (1.0 + np.exp(-logit))

x_test = Xv[:5]
with torch.no_grad():
    p_torch = torch.sigmoid(model(torch.from_numpy(x_test).to(device))).cpu().numpy()
p_numpy = numpy_forward(x_test, weights).squeeze()
print(f"\nParity check (PyTorch vs numpy):")
print(f"  torch: {p_torch[:5]}")
print(f"  numpy: {p_numpy[:5]}")
print(f"  max diff: {np.abs(p_torch - p_numpy).max():.6e}")

  l0_w: (64, 23), float32
  l0_b: (64,), float32
  l1_w: (32, 64), float32
  l1_b: (32,), float32
  l2_w: (1, 32), float32
  l2_b: (1,), float32

Saved /kaggle/working/weights.npz (16,058 bytes)

Parity check (PyTorch vs numpy):
  torch: [0.25269255 0.968809   0.8807569  0.9217171  0.82096934]
  numpy: [0.25269255 0.968809   0.8807569  0.9217171  0.82096934]
  max diff: 0.000000e+00


In [ ]:
# assemble main.py
v4_source = V4_LOCAL.read_text()
print(f"v4 source: {len(v4_source.splitlines()):,} lines, {len(v4_source):,} chars")

import re
v4_renamed = re.sub(
    r"^def agent\(([^)]*)\):",
    r"def _v4_agent_internal(\1):",
    v4_source, count=1, flags=re.MULTILINE,
)
assert "_v4_agent_internal" in v4_renamed, "failed to rename v4 agent"

# Hybrid wrapper code (validator + topk1)
HYBRID_CODE = '''
# ML Shot Validator + topk1 throttle 

import os as _os_h
from pathlib import Path as _Path_h
import numpy as _np_h
import math as _math_h

_VAL_THRESHOLD = 0.30  # the "sweet spot" found via threshold sweep
_VAL_TOP_K = 1

def _find_weights():
    cands = [
        _Path_h("/kaggle_simulations/agent/weights.npz"),
        _Path_h.cwd() / "weights.npz",
        _Path_h("weights.npz"),
    ]
    try:
        cands.insert(0, _Path_h(__file__).resolve().parent / "weights.npz")
    except NameError:
        pass
    for p in cands:
        if p.exists(): return p
    return None

_W_PATH = _find_weights()
_W = _np_h.load(_W_PATH) if _W_PATH is not None else None

def _validator_proba(x):
    if _W is None: return None
    h = _np_h.maximum(0, x @ _W["l0_w"].T + _W["l0_b"])
    h = _np_h.maximum(0, h @ _W["l1_w"].T + _W["l1_b"])
    logit = h @ _W["l2_w"].T + _W["l2_b"]
    return 1.0 / (1.0 + _np_h.exp(-logit))

_BOARD_H = 100.0; _MAX_SPEED_H = 6.0

def _fleet_speed_h(ships):
    if ships <= 0: return 1.0
    return 1.0 + (_MAX_SPEED_H - 1.0) * (_math_h.log(max(ships, 1)) / _math_h.log(1000.0)) ** 1.5

def _find_target_ray_h(src_xy, ang, planets, ray_horizon=200.0, perp_margin=1.0):
    sx, sy = src_xy; fx, fy = _math_h.cos(ang), _math_h.sin(ang)
    best_pid, best_perp = -1, 1e9
    for p in planets:
        pid, _, px, py, pr, _, _ = p
        pid = int(pid); px = float(px); py = float(py); pr = float(pr)
        dx = px - sx; dy = py - sy
        t = dx * fx + dy * fy
        if t <= 0 or t > ray_horizon: continue
        perp = abs(dx * fy - dy * fx)
        if perp <= pr + perp_margin and perp < best_perp:
            best_perp = perp; best_pid = pid
    return best_pid

def _encode_shot_h(obs, src_id, target_id, ships_sent):
    pdict = {int(p[0]): p for p in obs["planets"]}
    if src_id not in pdict or target_id not in pdict: return None
    src = pdict[src_id]; tgt = pdict[target_id]
    me = int(obs.get("player", 0))
    fleets = obs.get("fleets", [])
    planets = obs["planets"]
    my_t = sum(int(p[5]) for p in planets if int(p[1]) == me)
    en_t = sum(int(p[5]) for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    my_p = sum(1 for p in planets if int(p[1]) == me)
    en_p = sum(1 for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    sx, sy, sr, sships = float(src[2]), float(src[3]), float(src[4]), int(src[5])
    tx, ty, tr, tships = float(tgt[2]), float(tgt[3]), float(tgt[4]), int(tgt[5])
    sprod, tprod = float(src[6]), float(tgt[6])
    dx, dy = tx - sx, ty - sy
    dist = max(_math_h.hypot(dx, dy) - sr - tr, 0.0)
    speed = _fleet_speed_h(ships_sent); eta = dist / max(speed, 0.5)
    own_self = 1.0 if int(tgt[1]) == me else 0.0
    own_neutral = 1.0 if int(tgt[1]) < 0 else 0.0
    own_enemy = 1.0 if (int(tgt[1]) >= 0 and int(tgt[1]) != me) else 0.0
    sf = ships_sent / max(sships, 1)
    an = sum(1 for f in fleets if int(f[1]) == me)
    a_s = sum(int(f[6]) for f in fleets if int(f[1]) == me)
    en = sum(1 for f in fleets if int(f[1]) != me)
    e_s = sum(int(f[6]) for f in fleets if int(f[1]) != me)
    turn = int(obs.get("step", 0))
    return _np_h.array([
        sships/100.0, sprod/5.0, sr/4.0,
        tships/100.0, tprod/5.0, tr/4.0,
        own_self, own_neutral, own_enemy,
        ships_sent/100.0, sf,
        dist/_BOARD_H, eta/60.0, speed/_MAX_SPEED_H,
        an/10.0, a_s/100.0, en/10.0, e_s/100.0,
        turn/500.0, my_t/200.0, en_t/200.0,
        (my_t - en_t)/200.0,
        my_p/20.0, en_p/20.0,
    ], dtype=_np_h.float32)

def agent(obs, config=None):
    try:
        moves = _v4_agent_internal(obs, config)
    except TypeError:
        moves = _v4_agent_internal(obs)
    if not moves or _W is None:
        # If validator missing or v4 didn't propose anything, return raw v4 output
        return moves
    side = int(obs.get("player", 0))
    planets = obs["planets"]
    src_xy = {int(p[0]): (float(p[2]), float(p[3])) for p in planets}
    owner_by_id = {int(p[0]): int(p[1]) for p in planets}
    feats, idxs = [], []
    for i, mv in enumerate(moves):
        try:
            sid, ang, ships = int(mv[0]), float(mv[1]), int(mv[2])
        except Exception:
            continue
        if sid not in src_xy: continue
        tid = _find_target_ray_h(src_xy[sid], ang, planets)
        if tid < 0 or tid == sid: continue
        if owner_by_id.get(tid, -2) == side:
            continue  # never filter own-planet reinforcements
        f = _encode_shot_h(obs, sid, tid, ships)
        if f is None: continue
        feats.append(f); idxs.append(i)
    if feats:
        x = _np_h.stack(feats)
        probs = _validator_proba(x).squeeze(-1)
        keep = [True] * len(moves)
        for j, p in zip(idxs, probs):
            if p < _VAL_THRESHOLD:
                keep[j] = False
        moves = [m for k, m in enumerate(moves) if keep[k]]
    if not moves: return []
    # Keep only the largest ship-count moves. Change _VAL_TOP_K for topk sweeps.
    if _VAL_TOP_K and _VAL_TOP_K > 0 and len(moves) > _VAL_TOP_K:
        ranked = sorted(enumerate(moves), key=lambda im: int(im[1][2]), reverse=True)[:_VAL_TOP_K]
        keep_idx = {i for i, _ in ranked}
        moves = [m for i, m in enumerate(moves) if i in keep_idx]
    return moves
'''

# Concatenate 
main_py = v4_renamed + "\n\n" + HYBRID_CODE

(WORK / "main.py").write_text(main_py)
print(f"main.py: {(WORK/'main.py').stat().st_size:,} bytes ({len(main_py.splitlines()):,} lines)")

# Build top-k variants
TOPK_AGENT_PATHS = {}
for _k in (1, 2, 3):
    _variant = main_py.replace("_VAL_TOP_K = 1", f"_VAL_TOP_K = {_k}")
    _path = WORK / f"main_topk{_k}.py"
    _path.write_text(_variant)
    TOPK_AGENT_PATHS[_k] = _path
    print(f"{_path.name}: {_path.stat().st_size:,} bytes")
FINAL_AGENT_PATH = WORK / "main.py" 

v4 source: 368 lines, 14,999 chars
main.py: 20,290 bytes (504 lines)
main_topk1.py: 20,290 bytes
main_topk2.py: 20,290 bytes
main_topk3.py: 20,290 bytes


In [22]:
COMPACT_WRAPPER_FEATURE_BLOCK = r'''
_FEATURE_DIM_H = 23

def _encode_shot_h(obs, src_id, target_id, ships_sent):
    pdict = {int(p[0]): p for p in obs["planets"]}
    if src_id not in pdict or target_id not in pdict: return None
    src = pdict[src_id]; tgt = pdict[target_id]
    me = int(obs.get("player", 0))
    fleets = obs.get("fleets", [])
    planets = obs["planets"]
    my_t = 0.0; en_t = 0.0; my_prod = 0.0; en_prod = 0.0
    for p in planets:
        owner = int(p[1]); ships = float(p[5]); prod = float(p[6])
        if owner == me:
            my_t += ships; my_prod += prod
        elif owner >= 0:
            en_t += ships; en_prod += prod
    sx, sy, sr, sships = float(src[2]), float(src[3]), float(src[4]), int(src[5])
    tx, ty, tr, tships = float(tgt[2]), float(tgt[3]), float(tgt[4]), int(tgt[5])
    sprod, tprod = float(src[6]), float(tgt[6])
    dx, dy = tx - sx, ty - sy
    dist = max(_math_h.hypot(dx, dy) - sr - tr, 0.0)
    speed = _fleet_speed_h(ships_sent); eta = dist / max(speed, 0.5)
    own_neutral = 1.0 if int(tgt[1]) < 0 else 0.0
    own_enemy = 1.0 if (int(tgt[1]) >= 0 and int(tgt[1]) != me) else 0.0
    sf = ships_sent / max(sships, 1)
    an = 0; a_s = 0.0; en = 0; e_s = 0.0
    for f in fleets:
        if int(f[1]) == me:
            an += 1; a_s += float(f[6])
        else:
            en += 1; e_s += float(f[6])
    turn = int(obs.get("step", 0))
    total_ships = my_t + en_t
    total_prod = my_prod + en_prod
    target_growth = 0.0 if int(tgt[1]) == me else tprod * max(eta, 0.0)
    values = _np_h.array([
        sships/100.0, sprod/5.0, sr/4.0,
        tships/100.0, tprod/5.0, tr/4.0,
        own_neutral, own_enemy,
        ships_sent/100.0, sf,
        dist/_BOARD_H, eta/60.0,
        an/10.0, a_s/100.0, en/10.0, e_s/100.0,
        turn/500.0, en_t/200.0,
        (my_t - en_t)/200.0,
        tprod / max(total_prod, 1.0),
        my_t / max(total_ships, 1.0),
        ships_sent / max(tships + 1.0, 1.0),
        (ships_sent - (tships + target_growth)) / 100.0,
    ], dtype=_np_h.float32)
    if values.shape[0] != _FEATURE_DIM_H:
        raise ValueError(
            f"validator encoder produced {values.shape[0]} features; "
            f"expected {_FEATURE_DIM_H}"
        )
    return values
'''


def patch_compact_wrapper_features(agent_path):
    agent_path = Path(agent_path)
    text = agent_path.read_text()
    start = text.index("def _encode_shot_h(")
    end = text.index("\ndef agent(", start)
    patched = text[:start] + COMPACT_WRAPPER_FEATURE_BLOCK.strip() + text[end:]
    agent_path.write_text(patched)
    print(f"patched compact 23-dim wrapper features: {agent_path}")


patch_compact_wrapper_features(FINAL_AGENT_PATH)
for _path in TOPK_AGENT_PATHS.values():
    patch_compact_wrapper_features(_path)

with np.load(WORK / "weights.npz") as _check_weights:
    WEIGHTS_FEATURE_DIM = int(_check_weights["l0_w"].shape[1])

assert FEATURE_DIM == 23, f"train-time FEATURE_DIM is {FEATURE_DIM}, expected 23"
assert WEIGHTS_FEATURE_DIM == FEATURE_DIM, (
    f"weights.npz expects {WEIGHTS_FEATURE_DIM} features but the encoder uses "
    f"{FEATURE_DIM}. Rerun model creation, training, and weight export after Cell A."
)

print(
    "Wrapper patch complete:",
    f"train={FEATURE_DIM}, weights={WEIGHTS_FEATURE_DIM}, wrapper=23",
)


patched compact 23-dim wrapper features: /kaggle/working/main.py
patched compact 23-dim wrapper features: /kaggle/working/main_topk1.py
patched compact 23-dim wrapper features: /kaggle/working/main_topk2.py
patched compact 23-dim wrapper features: /kaggle/working/main_topk3.py
Wrapper patch complete: train=23, weights=23, wrapper=23


In [23]:
# Sanity check
from kaggle_environments import make

env = make("orbit_wars", debug=True)
result = env.run([str(FINAL_AGENT_PATH), OPPONENT_PATHS[0]])
final = env.steps[-1]
print(f"Game finished in {len(env.steps)} steps:")
for i, s in enumerate(final):
    print(f"  P{i}: reward={s.reward}, status={s.status}")

Game finished in 131 steps:
  P0: reward=-1, status=DONE
  P1: reward=1, status=DONE


## Top-K Experiment

Compares validator throttles `topk1`, `topk2`, and `topk3` against a small opponent/seed sample


In [35]:
import statistics
from kaggle_environments import make

def slot_metrics(env, slot):
    total_actions = 0
    active_turns = 0
    observable_turns = 0
    statuses = {}
    for step in env.steps:
        state = step[slot]
        statuses[state.status] = statuses.get(state.status, 0) + 1
        if state.observation is None:
            continue
        observable_turns += 1
        actions = state.action or []
        total_actions += len(actions)
        active_turns += int(bool(actions))
    reward = float(env.steps[-1][slot].reward or 0.0)
    return {
        "reward": reward,
        "steps": float(len(env.steps)),
        "moves_per_turn": total_actions / max(1, observable_turns),
        "idle_rate": 1.0 - active_turns / max(1, observable_turns),
        "invalid_turns": float(statuses.get("INVALID", 0)),
        "error_turns": float(statuses.get("ERROR", 0)),
        "timeout_turns": float(statuses.get("TIMEOUT", 0)),
    }

def summarize_rows(rows, label_key="agent"):
    for name in sorted({r[label_key] for r in rows}):
        selected = [r for r in rows if r[label_key] == name]
        wins = sum(float(r["reward"]) > 0 for r in selected)
        losses = sum(float(r["reward"]) < 0 for r in selected)
        draws = len(selected) - wins - losses
        print(
            f"{name}: W-L-D={wins}-{losses}-{draws} "
            f"win_rate={wins / max(1, len(selected)):.3f} "
            f"avg_steps={statistics.fmean(float(r['steps']) for r in selected):.1f} "
            f"moves/t={statistics.fmean(float(r['moves_per_turn']) for r in selected):.3f} "
            f"idle={statistics.fmean(float(r['idle_rate']) for r in selected):.3f} "
            f"invalid={sum(float(r['invalid_turns']) for r in selected):.0f} "
            f"errors={sum(float(r['error_turns']) for r in selected):.0f} "
            f"timeouts={sum(float(r['timeout_turns']) for r in selected):.0f}"
        )

RUN_TOPK_SWEEP = True
SWEEP_SEEDS = list(range(501, 516))
TOPK_SWEEP_VALUES = [1, 2, 3]
SWEEP_OPPONENTS = OPPONENT_PATHS[:min(2, len(OPPONENT_PATHS))]

topk_rows = []
if RUN_TOPK_SWEEP:
    for top_k in TOPK_SWEEP_VALUES:
        agent_path = TOPK_AGENT_PATHS[top_k]
        for opp in SWEEP_OPPONENTS:
            for seed in SWEEP_SEEDS:
                for labels in ((f"topk{top_k}", "opp"), ("opp", f"topk{top_k}")):
                    paths = [str(agent_path), opp] if labels[0].startswith("topk") else [opp, str(agent_path)]
                    env = make("orbit_wars", debug=False, configuration={"randomSeed": seed})
                    env.run(paths)
                    for slot, name in enumerate(labels):
                        if not name.startswith("topk"):
                            continue
                        row = {"top_k": top_k, "agent": name, "opponent": Path(opp).name, "seed": seed, "seat": slot}
                        row.update(slot_metrics(env, slot))
                        topk_rows.append(row)
                    print(f"topk={top_k} opp={Path(opp).name} seed={seed} seats={labels[0]}/{labels[1]} steps={len(env.steps)}", flush=True)
    summarize_rows(topk_rows)
else:
    print("RUN_TOPK_SWEEP=False; set it to True to run the experiment.")

topk=1 opp=buddy.py seed=501 seats=topk1/opp steps=113
topk=1 opp=buddy.py seed=501 seats=opp/topk1 steps=130
topk=1 opp=buddy.py seed=502 seats=topk1/opp steps=133
topk=1 opp=buddy.py seed=502 seats=opp/topk1 steps=150
topk=1 opp=buddy.py seed=503 seats=topk1/opp steps=171
topk=1 opp=buddy.py seed=503 seats=opp/topk1 steps=136
topk=1 opp=buddy.py seed=504 seats=topk1/opp steps=155
topk=1 opp=buddy.py seed=504 seats=opp/topk1 steps=138
topk=1 opp=buddy.py seed=505 seats=topk1/opp steps=140
topk=1 opp=buddy.py seed=505 seats=opp/topk1 steps=136
topk=1 opp=buddy.py seed=506 seats=topk1/opp steps=150
topk=1 opp=buddy.py seed=506 seats=opp/topk1 steps=153
topk=1 opp=buddy.py seed=507 seats=topk1/opp steps=110
topk=1 opp=buddy.py seed=507 seats=opp/topk1 steps=500
topk=1 opp=buddy.py seed=508 seats=topk1/opp steps=195
topk=1 opp=buddy.py seed=508 seats=opp/topk1 steps=149
topk=1 opp=buddy.py seed=509 seats=topk1/opp steps=122
topk=1 opp=buddy.py seed=509 seats=opp/topk1 steps=104
topk=1 opp

## Benchmark

In [30]:
import shutil

src = "/kaggle/input/datasets/cunmincut/0806-current-strongest/main.py"
dst = "/kaggle/working/main_original.py"

shutil.copy(src, dst)

'/kaggle/working/main_original.py'

In [34]:
import importlib.util
import sys
from pathlib import Path
from kaggle_environments import make

FINAL_AGENT_PATH = WORK / "main_topk2.py" 

def load_agent(path, module_name):
    path = Path(path)
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import agent from {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module.agent

def benchmark_pair(seed_start=901, seeds=5):
    agents = {
        "heuristic": load_agent(V4_LOCAL, "bench_orbitlite_heuristic"),
        "validator": load_agent(FINAL_AGENT_PATH, "bench_orbitlite_validator"),
    }
    rows = []
    for seed in range(seed_start, seed_start + seeds):
        for labels in (("heuristic", "validator"), ("validator", "heuristic")):
            env = make("orbit_wars", debug=False, configuration={"randomSeed": seed})
            env.run([agents[labels[0]], agents[labels[1]]])
            for slot, name in enumerate(labels):
                row = {"seed": seed, "seat": slot, "agent": name}
                row.update(slot_metrics(env, slot))
                rows.append(row)
            winner = max(range(2), key=lambda slot: float(env.steps[-1][slot].reward or 0.0))
            print(f"seed={seed} seats={labels[0]}/{labels[1]} winner={labels[winner]} steps={len(env.steps)}", flush=True)
    return rows

RUN_HEAD_TO_HEAD_BENCH = True
if RUN_HEAD_TO_HEAD_BENCH:
    bench_rows = benchmark_pair(seed_start=901, seeds=5)
    summarize_rows(bench_rows)
else:
    print("RUN_HEAD_TO_HEAD_BENCH=False; set it to True to run the benchmark.")

seed=901 seats=heuristic/validator winner=heuristic steps=110
seed=901 seats=validator/heuristic winner=heuristic steps=135
seed=902 seats=heuristic/validator winner=validator steps=140
seed=902 seats=validator/heuristic winner=heuristic steps=163
seed=903 seats=heuristic/validator winner=heuristic steps=94
seed=903 seats=validator/heuristic winner=heuristic steps=129
seed=904 seats=heuristic/validator winner=heuristic steps=149
seed=904 seats=validator/heuristic winner=validator steps=179
seed=905 seats=heuristic/validator winner=heuristic steps=194
seed=905 seats=validator/heuristic winner=validator steps=134
heuristic: W-L-D=7-3-0 win_rate=0.700 avg_steps=142.7 moves/t=1.070 idle=0.441 invalid=0 errors=0 timeouts=0
validator: W-L-D=3-7-0 win_rate=0.300 avg_steps=142.7 moves/t=0.748 idle=0.500 invalid=0 errors=0 timeouts=0


# Submission


In [32]:
import tarfile

FINAL_AGENT_PATH = WORK / "main_topk2.py" 

tar_path = WORK / "submission.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(FINAL_AGENT_PATH, arcname="main.py")
    tar.add(WORK / "weights.npz", arcname="weights.npz")
    for helper_file in sorted((WORK / "orbit_lite").glob("*.py")):
        tar.add(helper_file, arcname=f"orbit_lite/{helper_file.name}")

size = tar_path.stat().st_size
print(f"\nSubmission ready: {tar_path} ({size:,} bytes = {size/1024:.1f} KB)")
with tarfile.open(tar_path) as tar:
    for m in tar.getmembers():
        print(f"  {m.name}: {m.size:,} bytes")



Submission ready: /kaggle/working/submission.tar.gz (72,620 bytes = 70.9 KB)
  main.py: 20,687 bytes
  weights.npz: 16,058 bytes
  orbit_lite/__init__.py: 275 bytes
  orbit_lite/adapter.py: 8,441 bytes
  orbit_lite/aiming.py: 667 bytes
  orbit_lite/constants.py: 3,049 bytes
  orbit_lite/distance_cache.py: 4,772 bytes
  orbit_lite/garrison_launch.py: 18,568 bytes
  orbit_lite/geometry.py: 3,171 bytes
  orbit_lite/intercept_aim.py: 14,330 bytes
  orbit_lite/movement.py: 89,467 bytes
  orbit_lite/movement_aiming.py: 1,749 bytes
  orbit_lite/movement_step.py: 10,968 bytes
  orbit_lite/obs.py: 7,120 bytes
  orbit_lite/planner_core.py: 28,989 bytes


In [33]:
# Optional Kaggle CLI submit. Usually you can submit from the notebook UI after Save & Run All.
!kaggle competitions submit orbit-wars -f submission.tar.gz -m "orbitlite heuristic + ml validator ver 2"


100%|███████████████████████████████████████| 70.9k/70.9k [00:00<00:00, 336kB/s]
Successfully submitted to Orbit Wars